# Collecting data from web sources
Source: Wikipedia, *List of countries by exports* (text licensed CC BY-SA 4.0; the underlying figures are from the World Bank). If the internet is unavailable, run the *Offline fallback* cell and continue from *Parsing the real page*.

Each section matches a slide.

## A tiny HTML table

In [ ]:
sample = """
<table class="stats">
  <caption>Exports (USD m)</caption>
  <tr><th>Country</th><th>Exports</th></tr>
  <tr><td>Kenya</td><td>7,412</td></tr>
  <tr><td>Ghana</td><td>16,800</td></tr>
</table>
"""
print(len(sample), "characters of text")

## Parsing the sample

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(sample, "html.parser")
cap = soup.find("caption")
print(cap)
print(cap.get_text())
print(len(soup.find_all("tr")), "rows")

## Finding by attribute

In [ ]:
table = soup.find("table", class_="stats")
print(table["class"])
print(table.find("th").get_text())
print(soup.find("table", class_="other"))

## Extracting every cell

In [ ]:
for tr in soup.find_all("tr"):
    cells = []
    for c in tr.find_all(["th", "td"]):
        cells.append(c.get_text())
    print(cells)

## Checking robots.txt

In [ ]:
import requests
from urllib import robotparser

URL = ("https://en.wikipedia.org/wiki/"
       "List_of_countries_by_exports")
HEADERS = {"User-Agent":
           "TradeCourse/1.0 (me@example.org)"}
robots = requests.get(
    "https://en.wikipedia.org/robots.txt",
    headers=HEADERS, timeout=30)
rp = robotparser.RobotFileParser()
rp.parse(robots.text.splitlines())
print(rp.can_fetch("*", URL))

## Downloading the page

In [ ]:
resp = requests.get(URL, headers=HEADERS,
                    timeout=30)
print(resp.status_code)
print(resp.headers["Content-Type"])
html = resp.text
print(len(html), "characters")

## Parsing the real page

In [ ]:
soup = BeautifulSoup(html, "html.parser")
print(soup.title.get_text())
tables = soup.find_all("table")
print(len(tables), "tables on the page")
table = soup.find("table", class_="wikitable")
caption = table.find("caption")
print(caption.get_text(strip=True))

## Extracting the rows

In [ ]:
rows = []
for tr in table.find_all("tr")[1:]:
    tds = tr.find_all(["th", "td"])
    rows.append([c.get_text(strip=True)
                 for c in tds])
print(len(rows))
print(rows[0])
print(rows[1])

## Footnotes with a regular expression

In [ ]:
import re

print(re.sub(r"\[.*?\]", "", "2025[3]"))
print(re.sub(r"\[.*?\]", "", "Kenya[a]"))
print(re.sub(r"\[.*?\]", "", "1,234"))

## Into pandas, with types fixed

In [ ]:
import pandas as pd
df = pd.DataFrame(rows, columns=[
    "country", "exports_usd_m", "year",
    "top_export"])
# remove footnote markers such as [3]
df = df.replace(r"\[.*?\]", "", regex=True)
df["exports_usd_m"] = (
    df["exports_usd_m"]
    .str.replace(",", "")
    .astype(float))
df["year"] = df["year"].astype(int)
df.iloc[:, :3].head(3)

## The shortcut: read_html

In [ ]:
from io import StringIO

tables = pd.read_html(StringIO(html),
                      match="Exports")
print(len(tables), "matching tables")
tables[0].iloc[:, :3].head(3)

## Mixed reference years

In [ ]:
counts = df["year"].value_counts()
print(counts.sort_index())

## Offline fallback

In [ ]:
from pathlib import Path

CACHE = Path("../../data/cache")
cached = CACHE / "wikipedia_exports.html"
html = cached.read_text(encoding="utf-8")
print(len(html), "characters from cache")